# Base model with Bidirectional Encoder Representations from Transformers (BERT)

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import seaborn as sns
from wordcloud import WordCloud,STOPWORDS
import missingno as msno
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from keras.preprocessing import text
import keras
from keras.models import Sequential
from keras.layers import Dense,Embedding,LSTM,Dropout
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
from nltk import word_tokenize
from nltk.stem import PorterStemmer
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizer,BertTokenizerFast, BertForSequenceClassification
from transformers import TrainingArguments, Trainer, pipeline


2024-10-23 20:50:36.277280: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Prepare Dataset

Load JSON file

In [2]:
def load_json_file(filename):
    with open(filename) as f:
        file=json.load(f)
    return file   
    
filename="intents.json"
intents=load_json_file(filename)
print(intents)

{'intents': [{'tag': 'greeting', 'patterns': ['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Good day', 'Whats up'], 'responses': ['Hello!', 'Hi there, how can I help?'], 'context_set': ['']}, {'tag': 'name', 'patterns': ['what is your name', 'what should I call you', 'whats your name?'], 'responses': ['You can call me bot.', "I'm bot!", "I'm bot."], 'context_set': ['']}, {'tag': 'problem_solving', 'patterns': ['i need help', 'help me', 'tell me', 'i have some problem', 'problem', 'can you help me?'], 'responses': ['yes i can help you', 'tell me what can i do for you', 'what do you want me to do', 'how can i help you', 'what kind of help do you need', 'tell me'], 'context_set': ['']}, {'tag': 'jaundice', 'patterns': ['yellow skin', 'yellowish white part of eye', 'itching of the skin', 'light colored stools', 'dark colored urine'], 'responses': ['you might be suffering from jaundice'], 'context_set': ['']}, {'tag': 'emotional problems', 'patterns': ['anxiety', 'depression fatigue', 

Create dataframe for patterns and tags

In [3]:
def create_df():
    df=pd.DataFrame({'Pattern':[],'Tag':[]})
    return df
    
df=create_df()
df

,Pattern,Tag


Extract data from file

In [4]:
def extract_json_info(json_file,df):
    for intent in json_file['intents']:
        for pattern in intent['patterns']:
            sentence_tag=[pattern,intent['tag']]
            df.loc[len(df.index)] = sentence_tag
    return df        
df=extract_json_info(intents,df)

## Explore data

In [5]:
df.head()

,Pattern,Tag
0,Hi,greeting
1,How are you,greeting
2,Is anyone there?,greeting
3,Hello,greeting
4,Good day,greeting


In [6]:
df.shape

(244, 2)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 244 entries, 0 to 243
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Pattern  244 non-null    object
 1   Tag      244 non-null    object
dtypes: object(2)
memory usage: 5.7+ KB


In [8]:
x=len(df['Tag'].unique())
print(f"{x} classes in or tags")

print(df['Tag'].unique())

37 classes in or tags
['greeting' 'name' 'problem_solving' 'jaundice' 'emotional problems'
 'headache problems' 'appendicitis' 'anal cancer' 'allergy'
 'alcohol-related liver disease' 'bladder cancer' 'dehydration'
 'diabetic retinopathy' 'type 2 diabetes' 'type 1 diabetes'
 'Hypoglycaemia' 'Hyperglycaemia' 'diarrhoea' 'Ebola' 'kidney cancer'
 'glandular fever' 'hay fever' 'fibromyalgia' 'eye cancer' 'COVID-19'
 'common cold' 'chickenpox' 'stroke' 'breast problems'
 'reproductive health problems' 'lung problems' 'bladder problems'
 'eating or weight problems' 'influenza' 'asthma' 'hepatitis_B' 'Dengue']


In [9]:
df.isna().sum()

Pattern    0
Tag        0
dtype: int64

Understand the corpus

In [10]:
def get_corpus(series):
    words = []
    for text in series:
        for word in text.split():
            words.append(word.strip())
    return words

corpus = get_corpus(df.Pattern)
corpus[:5]
# print(corpus)

['Hi', 'How', 'are', 'you', 'Is']

In [11]:
from collections import Counter
counter = Counter(corpus)
most_common = counter.most_common(10)
print(most_common)
most_common = dict(most_common)
most_common

[('of', 37), ('or', 33), ('a', 25), ('and', 24), ('in', 19), ('the', 18), ('feeling', 18), ('loss', 17), ('pain', 17), ('vision', 12)]


{'of': 37,
 'or': 33,
 'a': 25,
 'and': 24,
 'in': 19,
 'the': 18,
 'feeling': 18,
 'loss': 17,
 'pain': 17,
 'vision': 12}

Create another dataframe for patterns, tags and labels

In [12]:
df2 = df.copy()
df2.head()

,Pattern,Tag
0,Hi,greeting
1,How are you,greeting
2,Is anyone there?,greeting
3,Hello,greeting
4,Good day,greeting


Create list of unique tags as labels

In [13]:
labels=df2['Tag'].unique().tolist()
labels=[s.strip() for s in labels]
labels[:5]

['greeting', 'name', 'problem_solving', 'jaundice', 'emotional problems']

Create `id` to `label` and vise versa

In [14]:
id2label={id:label for id,label in enumerate(labels)}
label2id={label:id for id,label in enumerate(labels)}
num_labels=len(labels)

In [15]:
id2label

{0: 'greeting',
 1: 'name',
 2: 'problem_solving',
 3: 'jaundice',
 4: 'emotional problems',
 5: 'headache problems',
 6: 'appendicitis',
 7: 'anal cancer',
 8: 'allergy',
 9: 'alcohol-related liver disease',
 10: 'bladder cancer',
 11: 'dehydration',
 12: 'diabetic retinopathy',
 13: 'type 2 diabetes',
 14: 'type 1 diabetes',
 15: 'Hypoglycaemia',
 16: 'Hyperglycaemia',
 17: 'diarrhoea',
 18: 'Ebola',
 19: 'kidney cancer',
 20: 'glandular fever',
 21: 'hay fever',
 22: 'fibromyalgia',
 23: 'eye cancer',
 24: 'COVID-19',
 25: 'common cold',
 26: 'chickenpox',
 27: 'stroke',
 28: 'breast problems',
 29: 'reproductive health problems',
 30: 'lung problems',
 31: 'bladder problems',
 32: 'eating or weight problems',
 33: 'influenza',
 34: 'asthma',
 35: 'hepatitis_B',
 36: 'Dengue'}

In [16]:
label2id

{'greeting': 0,
 'name': 1,
 'problem_solving': 2,
 'jaundice': 3,
 'emotional problems': 4,
 'headache problems': 5,
 'appendicitis': 6,
 'anal cancer': 7,
 'allergy': 8,
 'alcohol-related liver disease': 9,
 'bladder cancer': 10,
 'dehydration': 11,
 'diabetic retinopathy': 12,
 'type 2 diabetes': 13,
 'type 1 diabetes': 14,
 'Hypoglycaemia': 15,
 'Hyperglycaemia': 16,
 'diarrhoea': 17,
 'Ebola': 18,
 'kidney cancer': 19,
 'glandular fever': 20,
 'hay fever': 21,
 'fibromyalgia': 22,
 'eye cancer': 23,
 'COVID-19': 24,
 'common cold': 25,
 'chickenpox': 26,
 'stroke': 27,
 'breast problems': 28,
 'reproductive health problems': 29,
 'lung problems': 30,
 'bladder problems': 31,
 'eating or weight problems': 32,
 'influenza': 33,
 'asthma': 34,
 'hepatitis_B': 35,
 'Dengue': 36}

Add converted `label2id` to df2

In [17]:
df2['labels'] = df2['Tag'].map(lambda x: label2id[x.strip()])
df2.head()

,Pattern,Tag,labels
0,Hi,greeting,0
1,How are you,greeting,0
2,Is anyone there?,greeting,0
3,Hello,greeting,0
4,Good day,greeting,0


In [18]:
X=list(df2['Pattern'])
X[:5]

['Hi', 'How are you', 'Is anyone there?', 'Hello', 'Good day']

In [19]:
y=list(df2['labels'])
y[:5]

[0, 0, 0, 0, 0]

Define training and test datasets

In [20]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=100)

Define base model

In [21]:
model_name="bert-base-uncased"
max_len=256

Load tokenizer

In [22]:
tokenizer=BertTokenizer.from_pretrained(model_name,max_length=max_len)

/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Load model

In [23]:
model=BertForSequenceClassification.from_pretrained(model_name,num_labels=num_labels,id2label=id2label,label2id=label2id)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Define encoders

In [24]:
train_encoding=tokenizer(X_train,truncation=True,padding=True)
test_encoding=tokenizer(X_test,truncation=True,padding=True)

In [25]:
full_data = tokenizer(X, truncation=True, padding=True)

Data loader

In [26]:
class DataLoader(Dataset):
    
    def __init__(self, encodings, labels):
        
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
               
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):

        return len(self.labels)

In [27]:
train_dataloader = DataLoader(train_encoding, y_train)
test_dataloader = DataLoader(test_encoding, y_test)

In [28]:
fullDataLoader = DataLoader(full_data, y_test)

In [29]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids  # True labels
    preds = pred.predictions.argmax(-1)  # Predicted labels (argmax along the last axis)

    # Using precision_recall_fscore_support to compute precision, recall, F1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')

    # Using accuracy_score to compute accuracy
    acc = accuracy_score(labels, preds)
    
    # Returning a dictionary containing the computed metrics
    return {
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall
    }


Define training arguments

In [30]:
training_args = TrainingArguments(
    output_dir='./output', 
    do_train=True,
    do_eval=True,
    num_train_epochs=100,              
    per_device_train_batch_size=32,  
    per_device_eval_batch_size=16,
    warmup_steps=100,                
    weight_decay=0.05,
    logging_strategy='steps',
    logging_dir='./logs',            
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps", 
    load_best_model_at_end=True
)

/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,                 
    train_dataset=train_dataloader,         
    eval_dataset=test_dataloader,            
    compute_metrics= compute_metrics
)

In [32]:
%%time
trainer.train()

  0%|          | 0/600 [00:00<?, ?it/s]

{'loss': 3.5472, 'grad_norm': 6.391028881072998, 'learning_rate': 2.5e-05, 'epoch': 8.33}


  0%|          | 0/4 [00:00<?, ?it/s]

/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 3.5535542964935303, 'eval_Accuracy': 0.04918032786885246, 'eval_F1': 0.014430014430014432, 'eval_Precision': 0.008815426997245178, 'eval_Recall': 0.0404040404040404, 'eval_runtime': 0.4748, 'eval_samples_per_second': 128.469, 'eval_steps_per_second': 8.424, 'epoch': 8.33}
{'loss': 2.8781, 'grad_norm': 5.743937015533447, 'learning_rate': 5e-05, 'epoch': 16.67}


...

{'eval_loss': 4.152102470397949, 'eval_Accuracy': 0.19672131147540983, 'eval_F1': 0.1636904761904762, 'eval_Precision': 0.17559523809523808, 'eval_Recall': 0.20833333333333331, 'eval_runtime': 0.2044, 'eval_samples_per_second': 298.41, 'eval_steps_per_second': 19.568, 'epoch': 91.67}
{'loss': 0.2909, 'grad_norm': 2.3220505714416504, 'learning_rate': 0.0, 'epoch': 100.0}


  0%|          | 0/4 [00:00<?, ?it/s]

/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 4.167184352874756, 'eval_Accuracy': 0.19672131147540983, 'eval_F1': 0.15873015873015872, 'eval_Precision': 0.17027417027417027, 'eval_Recall': 0.20202020202020202, 'eval_runtime': 0.2169, 'eval_samples_per_second': 281.245, 'eval_steps_per_second': 18.442, 'epoch': 100.0}
{'train_runtime': 242.2419, 'train_samples_per_second': 75.544, 'train_steps_per_second': 2.477, 'train_loss': 0.9942811346054077, 'epoch': 100.0}
CPU times: user 3min 34s, sys: 15.2 s, total: 3min 49s
Wall time: 4min 2s


TrainOutput(global_step=600, training_loss=0.9942811346054077, metrics={'train_runtime': 242.2419, 'train_samples_per_second': 75.544, 'train_steps_per_second': 2.477, 'total_flos': 206956638702000.0, 'train_loss': 0.9942811346054077, 'epoch': 100.0})

Evaluate trainer

In [33]:
q=[trainer.evaluate(eval_dataset=df2) for df2 in [train_dataloader, test_dataloader]]
pd.DataFrame(q, index=["train","test"]).iloc[:,:5]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/johnmoses/miniconda3/envs/mconda38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


,eval_loss,eval_Accuracy,eval_F1,eval_Precision,eval_Recall
train,0.279341,0.868852,0.871538,0.912862,0.859588
test,4.128984,0.196721,0.163690,0.175595,0.208333


Save model and tokenizer

In [34]:
model_path = "base"

In [35]:
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

('base/tokenizer_config.json',
 'base/special_tokens_map.json',
 'base/vocab.txt',
 'base/added_tokens.json')

Load model and tokenizer

In [41]:
new_model = BertForSequenceClassification.from_pretrained(model_path)
new_tokenizer= BertTokenizerFast.from_pretrained(model_path)

In [37]:
pipe= pipeline("sentiment-analysis", model=new_model, tokenizer=new_tokenizer, device='mps')

In [38]:
pipe("Hello")

[{'label': 'greeting', 'score': 0.9569432139396667}]

## Predict

In [39]:
def predict(text):
    # Tokenize the input text and convert it to PyTorch tensors
    inputs = new_tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cpu")

    # Pass the input tensors through the model
    outputs = new_model(**inputs)

    # Softmax the output to get probabilities
    probs = outputs[0].softmax(1)

    # Get the predicted label index
    pred_label_idx = probs.argmax()

    # Convert the predicted label index to the actual label using model configuration
    pred_label = new_model.config.id2label[pred_label_idx.item()]

    return pred_label


In [42]:
text = "Hello"
predict(text)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


'greeting'

In [43]:
def model_response(text):
    response = ''
    score = pipe(text)[0]['score']
    tag = pipe(text)[0]['label']
    if score < 0.8:
        response += "Don't understand"
    else:
        label = new_model.config.label2id[pipe(text)[0]['label']]
        response = random.choice(intents['intents'][label]['responses'])
    return response, tag

In [45]:
model_response('hi')

('Hello!', 'greeting')